In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['hatch.linewidth'] = 0.2
import numpy as np
import pandas as pd
import pickle
from tqdm.notebook import tqdm
import polars as pl
import xgboost as xgb
print("xgboost version:", xgb.__version__)

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.file_locations import intermediate_files_location

from src.plot_helpers import make_histogram_plot

from src.ntuple_variables.variables import combined_training_vars

from src.df_helpers import lazy_height


# File Loading

In [ ]:
print("loading all_df.parquet...")
all_df = pl.scan_parquet(f"{intermediate_files_location}/all_df.parquet")
print(f"num events in all_df: {lazy_height(all_df)}")


In [ ]:
full_pred_data = all_df.filter(
    ~pl.col("filetype").is_in(["isotropic_one_gamma_overlay", "delete_one_gamma_overlay", "numuCC_rad_corrected", "NC_coherent_1g_reweighted"])
)

presel_merged_df_allvars = full_pred_data.filter(pl.col("wc_kine_reco_Enu") > 0)

presel_merged_df_allvars = presel_merged_df_allvars.filter(pl.col("normalizing_run_period") != "4a")


In [ ]:
presel_merged_df_allvars = presel_merged_df_allvars.with_columns([
    (
        (pl.col("wc_shw_sp_n_20mev_showers") > 0) &
        (pl.col("wc_reco_nuvtxX") > 5.0) & (pl.col("wc_reco_nuvtxX") < 250.0) &
        (pl.col("wc_single_photon_numu_score") > 0.4) &
        (pl.col("wc_single_photon_other_score") > 0.2) &
        (pl.col("wc_single_photon_ncpi0_score") > -0.05) &
        (pl.col("wc_single_photon_nue_score") > -1.0) &
        (pl.col("wc_shw_sp_n_20br1_showers") == 1)
    ).cast(pl.Int32).alias("erin_inclusive_1g_sel"),
    (
        (pl.col("wc_shw_sp_n_20mev_showers") > 0) &
        (pl.col("wc_reco_nuvtxX") > 5.0) & (pl.col("wc_reco_nuvtxX") < 250.0) &
        (pl.col("wc_single_photon_numu_score") > 0.1) &
        (pl.col("wc_single_photon_other_score") > -0.4) &
        (pl.col("wc_single_photon_ncpi0_score") < -0.4) &
        (pl.col("wc_single_photon_nue_score") > -20.0)
    ).cast(pl.Int32).alias("erin_nc_pi0_sideband_flag"),
])



# Preselection Histogram

In [ ]:
# load columns from presel_merged_df
load_vars = list(presel_merged_df_allvars.collect_schema().names())

# remove columns combined_training_vars variables, tons of variables that aren't needed
load_vars = [col for col in load_vars if not (col in combined_training_vars)]

load_vars += [col for col in load_vars if ("blip" in col) and (col not in load_vars)]

# TEMPORARY, since we didn't exclude all the pandora postprocessing variables
#load_vars = [col for col in load_vars if not (["pandora_max" in col])]

extra_vars = [
    "wc_kine_reco_Enu",
    "wc_reco_num_protons_35_MeV",
    "wc_reco_backwards_projected_dist",
    "wc_reco_distance_to_boundary",
    "wc_reco_shower_theta",
    "wc_reco_shower_phi",
    "wc_kine_pio_mass",
    "lantern_diphoton_mass",

    #"wc_reco_num_protons_35_MeV_75cm_from_reco_shower_vtx",
]

# add back in the current variable
for var in extra_vars:
    if var not in load_vars:
        load_vars.append(var)


In [ ]:
#load_vars

In [ ]:
presel_merged_df = presel_merged_df_allvars.select(load_vars).collect()

presel_merged_df

In [ ]:
erin_1g_sel_df = presel_merged_df.filter(pl.col("erin_inclusive_1g_sel") == 1)


In [ ]:
weights_df = pl.read_parquet(f"{intermediate_files_location}/presel_weights_df.parquet")


In [ ]:
detvar_df = pl.scan_parquet(f"{intermediate_files_location}/detvar_presel_df_train_vars.parquet")

detvar_df = detvar_df.with_columns([
    (
        (pl.col("wc_shw_sp_n_20mev_showers") > 0) &
        (pl.col("wc_reco_nuvtxX") > 5.0) & (pl.col("wc_reco_nuvtxX") < 250.0) &
        (pl.col("wc_single_photon_numu_score") > 0.4) &
        (pl.col("wc_single_photon_other_score") > 0.2) &
        (pl.col("wc_single_photon_ncpi0_score") > -0.05) &
        (pl.col("wc_single_photon_nue_score") > -1.0) &
        (pl.col("wc_shw_sp_n_20br1_showers") == 1)
    ).cast(pl.Int32).alias("erin_inclusive_1g_sel"),
    (
        (pl.col("wc_shw_sp_n_20mev_showers") > 0) &
        (pl.col("wc_reco_nuvtxX") > 5.0) & (pl.col("wc_reco_nuvtxX") < 250.0) &
        (pl.col("wc_single_photon_numu_score") > 0.1) &
        (pl.col("wc_single_photon_other_score") > -0.4) &
        (pl.col("wc_single_photon_ncpi0_score") < -0.4) &
        (pl.col("wc_single_photon_nue_score") > -20.0)
    ).cast(pl.Int32).alias("erin_nc_pi0_sideband_flag"),
])

detvar_load_vars = [col for col in load_vars if col not in ["wc_evtTimeNS_cor", "train_test_score", "will_use_for_50_50_training"]]


detvar_df = detvar_df.select(detvar_load_vars + ["vartype"]).collect()

erin_1g_sel_detvar_df = detvar_df.filter(pl.col("normalizing_run_period") != "4a").filter(pl.col("erin_inclusive_1g_sel") == 1)


In [ ]:
selection_var = "wc_kine_reco_Enu"
display_var = r"WC Reconstructed $E_\nu$ (GeV)"
bins = np.linspace(0, 2000, 21)
sel_title = "Run 4b WC Inclusive 1g Selection"
make_histogram_plot(pred_and_data_sel_df=presel_merged_df, weights_df=weights_df, 
            bins=bins, 
            var=selection_var, display_var=display_var, title=sel_title,
            include_ratio=True, include_decomposition=False,
            selname="generic_presel",
            dont_load_rw_from_systematic_cache=False, dont_load_detvar_from_systematic_cache=False, 
            use_rw_systematics=True, use_detvar_systematics=True, detvar_df=detvar_df,
            use_detvar_bootstrapping=True, num_bootstrap_rounds_detvar=100, num_bootstrap_samples_detvar=100,
            plot_sys_breakdown=True
            )


# Erin Inclusive 1g Selection Histograms

## Normal

In [ ]:
selection_var = "wc_reco_num_protons_35_MeV"
display_var = r"WC Reconstructed num protons (35 MeV threshold)"
bins = np.linspace(0, 4, 5)
sel_title = "Erin Inclusive 1g Selection"
make_histogram_plot(pred_and_data_sel_df=erin_1g_sel_df, weights_df=weights_df, 
            bins=bins, 
            var=selection_var, display_var=display_var, title=sel_title,
            include_ratio=True, include_decomposition=False,
            dont_load_rw_from_systematic_cache=False, dont_load_detvar_from_systematic_cache=False, 
            use_rw_systematics=True, use_detvar_systematics=True, detvar_df=erin_1g_sel_detvar_df,
            use_detvar_bootstrapping=True, num_bootstrap_rounds_detvar=100, num_bootstrap_samples_detvar=100,
            plot_sys_breakdown=True,
            data_type="4b open data",
            selname="erin_inclusive_1g_sel_new",
            breakdown_type="erin_categories",
            legend_fontsize=10, legend_ncol=1,
            ylim=[0, 50],
            yticks=np.linspace(0, 50, 11),
            include_overflow=False,
            )


## With CRT cut

In [ ]:
erin_1g_sel_crt_df = erin_1g_sel_df.filter(pl.col("pandora_crtveto") == 0)
erin_1g_sel_crt_detvar_df = erin_1g_sel_detvar_df.filter(pl.col("pandora_crtveto") == 0)


In [ ]:
selection_var = "wc_reco_num_protons_35_MeV"
display_var = r"WC Reconstructed num protons (35 MeV threshold)"
bins = np.linspace(0, 4, 5)
sel_title = "Erin Inclusive 1g Selection with CRT veto"
make_histogram_plot(pred_and_data_sel_df=erin_1g_sel_crt_df, weights_df=weights_df, 
            bins=bins, 
            var=selection_var, display_var=display_var, title=sel_title,
            include_ratio=True, include_decomposition=False,
            dont_load_rw_from_systematic_cache=False, dont_load_detvar_from_systematic_cache=False, 
            use_rw_systematics=True, use_detvar_systematics=True, detvar_df=erin_1g_sel_crt_detvar_df,
            use_detvar_bootstrapping=True, num_bootstrap_rounds_detvar=100, num_bootstrap_samples_detvar=100,
            plot_sys_breakdown=True,
            data_type="4b open data",
            selname="erin_inclusive_1g_sel_crt",
            breakdown_type="erin_categories",
            legend_fontsize=10, legend_ncol=1,
            ylim=[0, 50],
            yticks=np.linspace(0, 50, 11),
            include_overflow=False,
            )


## With WC 75 cm 35 MeV proton counting

In [ ]:
# TEMPORARY removed, not processed yet

"""
make_histogram_plot(pred_and_data_sel_df=erin_1g_sel_crt_df, bins=np.linspace(0, 4, 5), include_overflow=False,
            var="num_protons_35_MeV_75cm_from_reco_shower_vtx", display_var="WC Reconstructed num protons\n(35 MeV threshold, within 75 cm from shower vertex)", title="Erin Inclusive 1g Selection with CRT veto",
            selname="generic_presel",
            data_type="4b open data",
            breakdown_type="erin_categories",
            legend_fontsize=10, legend_ncol=1,
            ylim=[0, 50],
            yticks=np.linspace(0, 50, 11),
            )
"""


## With Blip-Enhanced 0p/Np split

In [ ]:
make_histogram_plot(pred_and_data_sel_df=erin_1g_sel_crt_df, bins=np.linspace(0, 20, 21), include_overflow=True,
            var="blip_backtrack_cones_n", display_var=r"blip_backtrack_cones_n", title="Erin Inclusive 1g Selection with CRT veto",
            selname="generic_presel",
            data_type="4b open data",
            breakdown_type="erin_Np0p_categories",
            legend_fontsize=10, legend_ncol=1,
            )
make_histogram_plot(pred_and_data_sel_df=erin_1g_sel_crt_df, bins=np.linspace(0, 20, 21), include_overflow=True,
            var="blip_no_shower_cone_no_backtrack_cones_n", display_var=r"blip_no_shower_cone_no_backtrack_cones_n", title="Erin Inclusive 1g Selection with CRT veto",
            selname="generic_presel",
            data_type="4b open data",
            breakdown_type="erin_Np0p_categories",
            legend_fontsize=10, legend_ncol=1,
            )
make_histogram_plot(pred_and_data_sel_df=erin_1g_sel_crt_df, bins=np.linspace(0, 20, 21), include_overflow=True,
            var="blip_sphere_n", display_var=r"blip_sphere_n", title="Erin Inclusive 1g Selection with CRT veto",
            selname="generic_presel",
            data_type="4b open data",
            breakdown_type="erin_Np0p_categories",
            legend_fontsize=10, legend_ncol=1,
            )
make_histogram_plot(pred_and_data_sel_df=erin_1g_sel_crt_df, bins=np.linspace(0, 20, 21), include_overflow=True,
            var="blip_no_shower_cone_n", display_var=r"blip_no_shower_cone_n", title="Erin Inclusive 1g Selection with CRT veto",
            selname="generic_presel",
            data_type="4b open data",
            breakdown_type="erin_Np0p_categories",
            legend_fontsize=10, legend_ncol=1,
            )
            

In [ ]:
# TODO: replace wc_reco_num_protons_35_MeV_plus_backtrack_blips with wc_reco_num_protons_35_MeV_75cm_from_reco_shower_vtx_plus_backtrack_blips
# TODO: replace "WC Reconstructed num protons (35 MeV) + backtrack blips" with "WC Reconstructed num protons (35 MeV, 75 cm) + backtrack blips"

selection_var = "wc_reco_num_protons_35_MeV_plus_backtrack_blips"
display_var = r"WC Reconstructed num protons (35 MeV) + backtrack blips"
bins = np.linspace(0, 4, 5)
sel_title = "Erin Inclusive 1g Selection with CRT veto"
make_histogram_plot(pred_and_data_sel_df=erin_1g_sel_crt_df, weights_df=weights_df, 
            bins=bins, 
            var=selection_var, display_var=display_var, title=sel_title,
            include_ratio=True, include_decomposition=False,
            dont_load_rw_from_systematic_cache=False, dont_load_detvar_from_systematic_cache=False, 
            use_rw_systematics=True, use_detvar_systematics=True, detvar_df=erin_1g_sel_crt_detvar_df,
            use_detvar_bootstrapping=True, num_bootstrap_rounds_detvar=100, num_bootstrap_samples_detvar=100,
            plot_sys_breakdown=True,
            data_type="4b open data",
            selname="erin_inclusive_1g_sel_crt",
            breakdown_type="erin_Np0p_categories",
            legend_fontsize=10, legend_ncol=1,
            ylim=[0, 50],
            yticks=np.linspace(0, 50, 11),
            include_overflow=False,
            )



## With Nn/0n split

In [ ]:
erin_1g_sel_crt_0n_df = erin_1g_sel_crt_df.filter((pl.col("blip_no_shower_cone_no_backtrack_cones_n") <= 2)
                                 & (pl.col("blip_no_shower_cone_no_backtrack_cones_sumE") <= 11))
erin_1g_sel_crt_Nn_df = erin_1g_sel_crt_df.filter((pl.col("blip_no_shower_cone_no_backtrack_cones_n") > 2)
                                 | (pl.col("blip_no_shower_cone_no_backtrack_cones_sumE") > 11))


erin_1g_sel_crt_0n_detvar_df = erin_1g_sel_crt_detvar_df.filter((pl.col("blip_no_shower_cone_no_backtrack_cones_n") <= 2)
                                 & (pl.col("blip_no_shower_cone_no_backtrack_cones_sumE") <= 11))
erin_1g_sel_crt_Nn_detvar_df = erin_1g_sel_crt_detvar_df.filter((pl.col("blip_no_shower_cone_no_backtrack_cones_n") > 2)
                                 | (pl.col("blip_no_shower_cone_no_backtrack_cones_sumE") > 11))


In [ ]:
# TODO: replace wc_reco_num_protons_35_MeV_plus_backtrack_blips with wc_reco_num_protons_35_MeV_75cm_from_reco_shower_vtx_plus_backtrack_blips
# TODO: replace "WC Reconstructed num protons (35 MeV) + backtrack blips" with "WC Reconstructed num protons (35 MeV, 75 cm) + backtrack blips"

selection_var = "wc_reco_num_protons_35_MeV_plus_backtrack_blips"
display_var = r"WC Reconstructed num protons (35 MeV) + backtrack blips"
bins = np.linspace(0, 4, 5)
sel_title = "Erin Inclusive 1g Selection with CRT veto and Nn blip cut"
make_histogram_plot(pred_and_data_sel_df=erin_1g_sel_crt_Nn_df, weights_df=weights_df, 
            bins=bins, 
            var=selection_var, display_var=display_var, title=sel_title,
            include_ratio=True, include_decomposition=False,
            dont_load_rw_from_systematic_cache=False, dont_load_detvar_from_systematic_cache=False, 
            use_rw_systematics=True, use_detvar_systematics=True, detvar_df=erin_1g_sel_crt_Nn_detvar_df,
            use_detvar_bootstrapping=True, num_bootstrap_rounds_detvar=100, num_bootstrap_samples_detvar=100,
            plot_sys_breakdown=True,
            data_type="4b open data",
            selname="erin_inclusive_1g_sel_crt_Nn",
            breakdown_type="erin_Np0pNn0n_categories",
            legend_fontsize=10, legend_ncol=1,
            ylim=[0, 50],
            yticks=np.linspace(0, 50, 11),
            include_overflow=False,
            )


selection_var = "wc_reco_num_protons_35_MeV_plus_backtrack_blips"
display_var = r"WC Reconstructed num protons (35 MeV) + backtrack blips"
bins = np.linspace(0, 4, 5)
sel_title = "Erin Inclusive 1g Selection with CRT veto and 0n blip cut"
make_histogram_plot(pred_and_data_sel_df=erin_1g_sel_crt_0n_df, weights_df=weights_df, 
            bins=bins, 
            var=selection_var, display_var=display_var, title=sel_title,
            include_ratio=True, include_decomposition=False,
            dont_load_rw_from_systematic_cache=False, dont_load_detvar_from_systematic_cache=False, 
            use_rw_systematics=True, use_detvar_systematics=True, detvar_df=erin_1g_sel_crt_0n_detvar_df,
            use_detvar_bootstrapping=True, num_bootstrap_rounds_detvar=100, num_bootstrap_samples_detvar=100,
            plot_sys_breakdown=True,
            data_type="4b open data",
            selname="erin_inclusive_1g_sel_crt_Nn",
            breakdown_type="erin_Np0pNn0n_categories",
            legend_fontsize=10, legend_ncol=1,
            ylim=[0, 50],
            yticks=np.linspace(0, 50, 11),
            include_overflow=False,
            )

In [ ]:
# TODO: replace wc_reco_num_protons_35_MeV_plus_backtrack_blips with wc_reco_num_protons_35_MeV_75cm_from_reco_shower_vtx_plus_backtrack_blips
# TODO: replace "WC Reconstructed num protons (35 MeV) + backtrack blips" with "WC Reconstructed num protons (35 MeV, 75 cm) + backtrack blips"

selection_var = "wc_reco_num_protons_35_MeV_plus_backtrack_blips"
display_var = r"WC Reconstructed num protons (35 MeV) + backtrack blips"
bins = np.linspace(0, 4, 5)
sel_title = "Erin Inclusive 1g Selection with CRT veto and Nn blip cut"
make_histogram_plot(pred_and_data_sel_df=erin_1g_sel_crt_Nn_df, weights_df=weights_df, 
            bins=bins, 
            var=selection_var, display_var=display_var, title=sel_title,
            include_ratio=True, include_decomposition=False,
            dont_load_rw_from_systematic_cache=False, dont_load_detvar_from_systematic_cache=False, 
            use_rw_systematics=True, use_detvar_systematics=True, detvar_df=erin_1g_sel_crt_Nn_detvar_df,
            use_detvar_bootstrapping=True, num_bootstrap_rounds_detvar=100, num_bootstrap_samples_detvar=100,
            plot_sys_breakdown=True,
            data_type="4b open data",
            selname="erin_inclusive_1g_sel_crt_Nn",
            breakdown_type="erin_Np0pNn0n_categories",
            legend_fontsize=10, legend_ncol=1,
            include_overflow=False,
            )


selection_var = "wc_reco_num_protons_35_MeV_plus_backtrack_blips"
display_var = r"WC Reconstructed num protons (35 MeV) + backtrack blips"
bins = np.linspace(0, 4, 5)
sel_title = "Erin Inclusive 1g Selection with CRT veto and 0n blip cut"
make_histogram_plot(pred_and_data_sel_df=erin_1g_sel_crt_0n_df, weights_df=weights_df, 
            bins=bins, 
            var=selection_var, display_var=display_var, title=sel_title,
            include_ratio=True, include_decomposition=False,
            dont_load_rw_from_systematic_cache=False, dont_load_detvar_from_systematic_cache=False, 
            use_rw_systematics=True, use_detvar_systematics=True, detvar_df=erin_1g_sel_crt_0n_detvar_df,
            use_detvar_bootstrapping=True, num_bootstrap_rounds_detvar=100, num_bootstrap_samples_detvar=100,
            plot_sys_breakdown=True,
            data_type="4b open data",
            selname="erin_inclusive_1g_sel_crt_Nn",
            breakdown_type="erin_Np0pNn0n_categories",
            legend_fontsize=10, legend_ncol=1,
            include_overflow=False,
            )


# Erin Sideband Histograms

In [ ]:
erin_ncpi0_sel_df = presel_merged_df.filter(pl.col("erin_nc_pi0_sideband_flag") == 1)
erin_ncpi0_sel_crt_df = erin_ncpi0_sel_df.filter(pl.col("pandora_crtveto") == 0)

erin_ncpi0_sel_crt_0n_df = erin_ncpi0_sel_crt_df.filter((pl.col("blip_no_shower_cone_no_backtrack_cones_n") <= 2)
                                 & (pl.col("blip_no_shower_cone_no_backtrack_cones_sumE") <= 11))
erin_ncpi0_sel_crt_Nn_df = erin_ncpi0_sel_crt_df.filter((pl.col("blip_no_shower_cone_no_backtrack_cones_n") > 2)
                                 | (pl.col("blip_no_shower_cone_no_backtrack_cones_sumE") > 11))

erin_ncpi0_sel_detvar_df = detvar_df.filter(pl.col("erin_nc_pi0_sideband_flag") == 1)
erin_ncpi0_sel_crt_0n_detvar_df = erin_ncpi0_sel_detvar_df.filter((pl.col("blip_no_shower_cone_no_backtrack_cones_n") <= 2)
                                 & (pl.col("blip_no_shower_cone_no_backtrack_cones_sumE") <= 11))
erin_ncpi0_sel_crt_Nn_detvar_df = erin_ncpi0_sel_detvar_df.filter((pl.col("blip_no_shower_cone_no_backtrack_cones_n") > 2)
                                 | (pl.col("blip_no_shower_cone_no_backtrack_cones_sumE") > 11))


In [ ]:
# TODO: replace wc_reco_num_protons_35_MeV_plus_backtrack_blips with wc_reco_num_protons_35_MeV_75cm_from_reco_shower_vtx_plus_backtrack_blips
# TODO: replace "WC Reconstructed num protons (35 MeV) + backtrack blips" with "WC Reconstructed num protons (35 MeV, 75 cm) + backtrack blips"

selection_var = "wc_reco_num_protons_35_MeV_plus_backtrack_blips"
display_var = r"WC Reconstructed num protons (35 MeV) + backtrack blips"
bins = np.linspace(0, 4, 5)
sel_title = "Erin NC Pi0 Sideband with CRT veto and Nn blip cut"
make_histogram_plot(pred_and_data_sel_df=erin_ncpi0_sel_crt_Nn_df, weights_df=weights_df, 
            bins=bins, 
            var=selection_var, display_var=display_var, title=sel_title,
            include_ratio=True, include_decomposition=False,
            dont_load_rw_from_systematic_cache=False, dont_load_detvar_from_systematic_cache=False, 
            use_rw_systematics=True, use_detvar_systematics=True, detvar_df=erin_ncpi0_sel_crt_Nn_detvar_df,
            use_detvar_bootstrapping=True, num_bootstrap_rounds_detvar=100, num_bootstrap_samples_detvar=100,
            plot_sys_breakdown=True,
            data_type="4b open data",
            selname="erin_ncpi0_sel_crt_Nn",
            breakdown_type="erin_categories",
            legend_fontsize=10, legend_ncol=1,
            ylim=[0, 240],
            yticks=np.linspace(0, 240, 13),
            include_overflow=False,
            )

selection_var = "wc_reco_num_protons_35_MeV_plus_backtrack_blips"
display_var = r"WC Reconstructed num protons (35 MeV) + backtrack blips"
bins = np.linspace(0, 4, 5)
sel_title = "Erin NC Pi0 Sideband with CRT veto and 0n blip cut"
make_histogram_plot(pred_and_data_sel_df=erin_ncpi0_sel_crt_0n_df, weights_df=weights_df, 
            bins=bins, 
            var=selection_var, display_var=display_var, title=sel_title,
            include_ratio=True, include_decomposition=False,
            dont_load_rw_from_systematic_cache=False, dont_load_detvar_from_systematic_cache=False, 
            use_rw_systematics=True, use_detvar_systematics=True, detvar_df=erin_ncpi0_sel_crt_0n_detvar_df,
            use_detvar_bootstrapping=True, num_bootstrap_rounds_detvar=100, num_bootstrap_samples_detvar=100,
            plot_sys_breakdown=True,
            data_type="4b open data",
            selname="erin_ncpi0_sel_crt_0n",
            breakdown_type="erin_categories",
            legend_fontsize=10, legend_ncol=1,
            ylim=[0, 240],
            yticks=np.linspace(0, 240, 13),
            include_overflow=False,
            )



In [ ]:
# TODO: replace wc_reco_num_protons_35_MeV_plus_backtrack_blips with wc_reco_num_protons_35_MeV_75cm_from_reco_shower_vtx_plus_backtrack_blips
# TODO: replace "WC Reconstructed num protons (35 MeV) + backtrack blips" with "WC Reconstructed num protons (35 MeV, 75 cm) + backtrack blips"

selection_var = "wc_reco_num_protons_35_MeV_plus_backtrack_blips"
display_var = r"WC Reconstructed num protons (35 MeV) + backtrack blips"
bins = np.linspace(0, 4, 5)
sel_title = "Erin NC Pi0 Sideband with CRT veto and Nn blip cut"
make_histogram_plot(pred_and_data_sel_df=erin_ncpi0_sel_crt_Nn_df, weights_df=weights_df, 
            bins=bins, 
            var=selection_var, display_var=display_var, title=sel_title,
            include_ratio=True, include_decomposition=False,
            dont_load_rw_from_systematic_cache=False, dont_load_detvar_from_systematic_cache=False, 
            use_rw_systematics=True, use_detvar_systematics=True, detvar_df=erin_ncpi0_sel_crt_Nn_detvar_df,
            use_detvar_bootstrapping=True, num_bootstrap_rounds_detvar=100, num_bootstrap_samples_detvar=100,
            plot_sys_breakdown=True,
            data_type="4b open data",
            selname="erin_ncpi0_sel_crt_Nn",
            breakdown_type="erin_Np0pNn0n_pi0_categories",
            legend_fontsize=10, legend_ncol=1,
            include_overflow=False,
            )

selection_var = "wc_reco_num_protons_35_MeV_plus_backtrack_blips"
display_var = r"WC Reconstructed num protons (35 MeV) + backtrack blips"
bins = np.linspace(0, 4, 5)
sel_title = "Erin NC Pi0 Sideband with CRT veto and 0n blip cut"
make_histogram_plot(pred_and_data_sel_df=erin_ncpi0_sel_crt_0n_df, weights_df=weights_df, 
            bins=bins, 
            var=selection_var, display_var=display_var, title=sel_title,
            include_ratio=True, include_decomposition=False,
            dont_load_rw_from_systematic_cache=False, dont_load_detvar_from_systematic_cache=False, 
            use_rw_systematics=True, use_detvar_systematics=True, detvar_df=erin_ncpi0_sel_crt_0n_detvar_df,
            use_detvar_bootstrapping=True, num_bootstrap_rounds_detvar=100, num_bootstrap_samples_detvar=100,
            plot_sys_breakdown=True,
            data_type="4b open data",
            selname="erin_ncpi0_sel_crt_0n",
            breakdown_type="erin_Np0pNn0n_pi0_categories",
            legend_fontsize=10, legend_ncol=1,
            include_overflow=False,
            )